In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
import plotly.express as px

In [9]:
df = pd.read_csv("RAW_DEMO.csv")
df.head()

,pl_name,hostname,default_flag,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,soltype,pl_controv_flag,...,ra,decstr,dec,sy_dist,sy_vmag,sy_kmag,sy_gaiamag,rowupdate,pl_pubdate,releasedate
0,TRAPPIST-1 h,TRAPPIST-1,0,1,7,Transit,2017,Multiple Observatories,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.02,10.296,15.6451,11/14/2022,2020-08,11/14/2022
1,TRAPPIST-1 h,TRAPPIST-1,0,1,7,Transit,2017,Multiple Observatories,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.02,10.296,15.6451,12/10/2020,2018-05,12/17/2020
2,TRAPPIST-1 h,TRAPPIST-1,0,1,7,Transit,2017,Multiple Observatories,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.02,10.296,15.6451,6/13/2017,2017-06,6/15/2017
3,TRAPPIST-1 h,TRAPPIST-1,1,1,7,Transit,2017,Multiple Observatories,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.02,10.296,15.6451,11/14/2022,2021-02,11/14/2022
4,TRAPPIST-1 h,TRAPPIST-1,0,1,7,Transit,2017,Multiple Observatories,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.02,10.296,15.6451,6/13/2017,2017-02,2/22/2017


Note: default_flag is 1 for the accepted values of a planet and 0 for all other values of a planet, filter by default_flag==1 to get one of each planet

In [10]:
# Filter to have one of each planet
df = df.drop(df[df.default_flag == 0].index)

In [11]:
df

,pl_name,hostname,default_flag,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,soltype,pl_controv_flag,...,ra,decstr,dec,sy_dist,sy_vmag,sy_kmag,sy_gaiamag,rowupdate,pl_pubdate,releasedate
3,TRAPPIST-1 h,TRAPPIST-1,1,1,7,Transit,2017,Multiple Observatories,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.020,10.296,15.64510,11/14/2022,2021-02,11/14/2022
7,TRAPPIST-1 g,TRAPPIST-1,1,1,7,Transit,2017,Multiple Observatories,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.020,10.296,15.64510,11/14/2022,2021-02,11/14/2022
13,TRAPPIST-1 f,TRAPPIST-1,1,1,7,Transit,2017,Multiple Observatories,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.020,10.296,15.64510,11/14/2022,2021-02,11/14/2022
16,TRAPPIST-1 e,TRAPPIST-1,1,1,7,Transit,2017,Multiple Observatories,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.020,10.296,15.64510,11/14/2022,2021-02,11/14/2022
23,TRAPPIST-1 d,TRAPPIST-1,1,1,7,Transit,2016,La Silla Observatory,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.020,10.296,15.64510,11/14/2022,2021-02,11/14/2022
24,TRAPPIST-1 c,TRAPPIST-1,1,1,7,Transit,2016,La Silla Observatory,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.020,10.296,15.64510,11/14/2022,2021-02,11/14/2022
32,TRAPPIST-1 b,TRAPPIST-1,1,1,7,Transit,2016,La Silla Observatory,Published Confirmed,0,...,346.626392,-05d02m36.46s,-5.043462,NaN,17.020,10.296,15.64510,11/14/2022,2021-02,11/14/2022
40,HIP 65426 b,HIP 65426,1,1,1,Imaging,2017,Paranal Observatory,Published Confirmed,0,...,201.150173,-51d30m16.34s,-51.504538,108.8750,7.010,6.771,6.98273,7/11/2017,2017-09,7/13/2017
43,TYC 8998-760-1 b,TYC 8998-760-1,1,1,2,Imaging,2020,Paranal Observatory,Published Confirmed,0,...,201.300110,-64d56m20.96s,-64.939157,94.6177,11.124,8.392,10.65050,4/30/2026,2025-11,4/30/2026
45,TYC 8998-760-1 c,TYC 8998-760-1,1,1,2,Imaging,2020,Paranal Observatory,Published Confirmed,0,...,201.300110,-64d56m20.96s,-64.939157,94.6177,11.124,8.392,10.65050,7/17/2020,2020-07,7/23/2020


In [12]:
len(df["pl_rade"].dropna()) / len(df["pl_rade"])

0.9

In [13]:
# GET TARGET (RADIUS)
df = df.dropna(subset=['pl_rade'])

y = np.log10(df["pl_rade"])
df = df.drop(["pl_rade", "pl_radj"], axis=1)

In [14]:
# DATE
releasedate = pd.to_datetime(df["releasedate"], format="mixed")
df = df.drop("releasedate", axis=1)
df["release_year"] = releasedate.dt.year
df["release_month"] = releasedate.dt.month

In [15]:
# PREPROCESS MASS DATA

mass_column_names = ['pl_masse',
 'pl_masseerr1',
 'pl_masseerr2',
 'pl_masselim',
 'pl_massj',
 'pl_massjerr1',
 'pl_massjerr2',
 'pl_massjlim',
 'pl_msinie',
 'pl_msinieerr1',
 'pl_msinieerr2',
 'pl_msinielim',
 'pl_msinij',
 'pl_msinijerr1',
 'pl_msinijerr2',
 'pl_msinijlim',
 'pl_cmasse',
 'pl_cmasseerr1',
 'pl_cmasseerr2',
 'pl_cmasselim',
 'pl_cmassj',
 'pl_cmassjerr1',
 'pl_cmassjerr2',
 'pl_cmassjlim',
 'pl_bmasse',
 'pl_bmasseerr1',
 'pl_bmasseerr2',
 'pl_bmasselim',
 'pl_bmassj',
 'pl_bmassjerr1',
 'pl_bmassjerr2',
 'pl_bmassjlim',
 'pl_bmassprov',
 'pl_dens']

mass_columns = {"Mass":"pl_masse", "Msini":"pl_msinie", "Msin(i)/sin(i)":"pl_cmasse"}

mass = []
mass_error_width = []
mass_type = []

for i, row in df.iterrows():
    t = row["pl_bmassprov"]
    if t in mass_columns:
        m = row[mass_columns[t]]
        merr1 = row[mass_columns[t] + "err1"]
        merr2 = row[mass_columns[t] + "err2"]
        
        mass.append(m)
        mass_type.append(t)
        mass_error_width.append(abs(merr1)+abs(merr2))
    else:
        mass.append(np.NaN)
        mass_type.append(np.NaN)
        mass_error_width.append(np.NaN)

df["pl_mass"] = mass
df["pl_mass_type"] = mass_type

df = df.drop(mass_column_names, axis=1)

KeyError: 'pl_masse'

In [16]:
to_drop = ['default_flag', "pl_letter", "sy_mnum", "rowupdate", "decstr", "rastr", "disc_pubdate", "pl_pubdate", "discoverymethod", "disc_instrument", "disc_facility", "disc_locale"]
for s in df.columns:
    if s.endswith("err1") or s.endswith("err2") or s.endswith("lim") or s.endswith("name") or s.endswith("id"):
        to_drop.append(s)
    if s in ['rv_flag', 'pul_flag', 'ptv_flag', 'tran_flag','ast_flag','obm_flag','micro_flag','etv_flag','ima_flag','dkin_flag']: # duplicates from discovery method
        to_drop.append(s)
X = df.drop(to_drop, axis=1)

KeyError: "['pl_letter', 'sy_mnum', 'disc_pubdate', 'disc_instrument', 'disc_locale'] not found in axis"

In [9]:
cols = list(X.columns)
percent_nan = []

for col in cols:
    percent_nan.append(len(X[col].dropna()) / len(X[col]))

nan_ratio = pd.DataFrame()
nan_ratio["Feature"] = cols
nan_ratio["not NaN"] = percent_nan
nan_ratio = nan_ratio.sort_values("not NaN", ascending=False)
#print(nan_ratio.to_string())

In [10]:
cutoff_ratio = 0.8
keep = nan_ratio["Feature"][nan_ratio["not NaN"] > cutoff_ratio].values
X = X[keep]
X = X.dropna()

In [11]:
X = X[X["disc_telescope"] == "0.95 m Kepler Telescope"]
X = X.drop("disc_telescope", axis=1)

In [12]:
X_number = X.select_dtypes(include='number')
X_categorical = X.select_dtypes(exclude='number')

In [13]:
for col in X_categorical.columns:
    print(col)
    print(X_categorical[col].value_counts())
    print("\n\n")

soltype
soltype
Kepler Project Candidate (q1_q16_koi)         2482
Kepler Project Candidate (q1_q17_dr24_koi)    2464
Kepler Project Candidate (q1_q12_koi)         2445
Kepler Project Candidate (q1_q17_dr25_koi)    2421
Kepler Project Candidate (q1_q8_koi)          2109
Published Confirmed                           2045
Published Candidate                           1385
Name: count, dtype: int64





In [ ]:
X = pd.concat([X_number, pd.get_dummies(X_categorical)], axis=1)
data = X
data["log radius"] = y

In [ ]:
data.to_csv("CleanPreprocessedExoplanet.csv")